# Taller 1: Econometría

- **Profesor:** Francisco Alfaro Medina
- **Ayudantes:** Krischnna Cortez y Karen Rojas


### Instrucciones

- Dispone de **60 minutos** para completar los **100 puntos** del taller.
- Cuide la presentación y redacción de sus respuestas.
- Puede utilizar su computador y los apuntes de clase y ayudantía.
- Debe entregar un archivo **PDF** y un **R script** (extensión `.R`).

> ⚠️ **Importante para Colab:** Este notebook usa un kernel de R. Si abre este archivo en Google Colab, seleccione **Runtime → Change runtime type → R** antes de ejecutar cualquier celda.

---
# Sección 1: Datos Aleatorios *(30 puntos)*

En esta sección trabajaremos con un dataset **simulado** de 50 alumnos de la USM. El dataset contiene las siguientes variables:

| Variable | Descripción |
|---|---|
| `hrs_sueno` | Horas de sueño promedio en el último mes |
| `profesor_part` | Si recibió ayuda de profesor particular (0/1) |
| `media_sem_pasado` | Promedio de notas del semestre anterior |
| `tiempo_est` | Horas de estudio dedicadas |
| `asistencia` | Porcentaje de asistencia a clases |
| `nivel_socioec` | Nivel socioeconómico (1 al 5) |
| `notas` | **Variable dependiente** — nota del alumno |

## Pregunta 1.1 — Generar el dataset *(6 pts.)*

Antes de ejecutar el código, **cambie la semilla** según la primera letra de su apellido:

| A–E | F–J | K–O | P–T | U–Z |
|:---:|:---:|:---:|:---:|:---:|
| 123 | 456 | 789 | 101112 | 131415 |

Reemplace el valor en `set.seed(...)` antes de continuar.

In [2]:
# -------------------------------------------------------
# Pregunta 1.1: Generar el dataframe "datos"
# Cambie la semilla según la primera letra de su apellido
# A-E: 123 | F-J: 456 | K-O: 789 | P-T: 101112 | U-Z: 131415
# -------------------------------------------------------

set.seed(131415)  # <-- CAMBIE ESTE VALOR SEGÚN SU APELLIDO

datos <- data.frame(
  hrs_sueno        = round(runif(50, min = 5,  max = 10),  1),
  profesor_part    = sample(c(0, 1), 50, replace = TRUE),
  media_sem_pasado = round(runif(50, min = 60, max = 100), 1),
  tiempo_est       = round(runif(50, min = 1,  max = 8),   1),
  asistencia       = round(runif(50, min = 60, max = 100), 1),
  nivel_socioec    = sample(1:5, 50, replace = TRUE)
)

# Calcular notas con ponderaciones definidas
datos$notas <- 30 +
  datos$hrs_sueno        * 1.5  +
  datos$profesor_part    * 3    +
  datos$media_sem_pasado * 0.2  +
  datos$tiempo_est       * 2    +
  datos$asistencia       * 0.15 +
  datos$nivel_socioec    * 2    +
  rnorm(50, mean = 0, sd = 5)

# Asegurar rango entre 20 y 100
datos$notas <- pmax(pmin(datos$notas, 100), 20)

# Vista rápida del dataset
cat("Dimensiones del dataset:", nrow(datos), "filas x", ncol(datos), "columnas\n")
head(datos)

Dimensiones del dataset: 50 filas x 7 columnas


,hrs_sueno,profesor_part,media_sem_pasado,tiempo_est,asistencia,nivel_socioec,notas
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<dbl>
1,6.8,0,65.6,1.6,88.3,5,83.27466
2,6.7,0,98.7,5.0,70.8,5,92.21466
3,9.3,1,72.6,7.4,98.8,3,99.53520
4,8.2,1,98.7,7.8,62.1,2,86.35777
5,9.3,0,71.8,5.3,98.9,4,92.87862
6,7.9,1,62.3,6.2,83.3,5,95.64333


## Pregunta 1.2 — Redondear notas *(2 pts.)*

Redondee la variable `notas` a **1 decimal**.

In [3]:
#Pregunta 1.2: Redondear notas a decimal
datos$notas <- round(datos$notas, 1)
head(datos$notas)

[1] 83.3 92.2 99.5 86.4 92.9 95.6

## Pregunta 1.3 — Estimadores β via álgebra matricial *(8 pts.)*

Calcule los estimadores MCO **manualmente**, usando la fórmula matricial:

$$\hat{\boldsymbol{\beta}} = (\mathbf{X}^\top \mathbf{X})^{-1} \mathbf{X}^\top \mathbf{y}$$

**Sin usar** la función `lm()` de R.

In [8]:
# Pregunta 1.3: Estimadores beta via álgebra matricial (sin lm)
Y <- as.matrix(datos$notas)
X <- as.matrix(cbind(1,datos[, c("hrs_sueno","profesor_part","media_sem_pasado","tiempo_est","asistencia","nivel_socioec")]))
beta_hat <- solve(t(X)%*% X) %*% t(X) %*% Y
rownames(beta_hat) <- c("Beta_cero","hrs_sueno","profesor_part","media_sem_pasado","tiempo_est","asistencia","nivel_socioec")
cat("Estimadores beta calculados manualmente:\n")
print(beta_hat)

Estimadores beta calculados manualmente:
                       [,1]
Beta_cero        27.6664510
hrs_sueno         0.9166767
profesor_part     2.5179959
media_sem_pasado  0.2034632
tiempo_est        1.8507803
asistencia        0.2383570
nivel_socioec     2.0566217


## Pregunta 1.4 — Modelo con `lm()` e interpretación *(8 pts.)*

Genere el modelo de regresión múltiple usando la función `lm()` y obtenga el resumen con `summary()`.  
Luego, **interprete cada coeficiente** en el espacio indicado.

> 💡 **Tip:** Los β de `lm()` deben coincidir con los calculados manualmente en la pregunta anterior.

In [10]:
# Pregunta 1.4: Modelo con lm() y summary
modelo_lm <- lm(notas~hrs_sueno + profesor_part + media_sem_pasado + tiempo_est + asistencia + nivel_socioec, data =datos)
summary(modelo_lm)


Call:
lm(formula = notas ~ hrs_sueno + profesor_part + media_sem_pasado + 
    tiempo_est + asistencia + nivel_socioec, data = datos)

Residuals:
    Min      1Q  Median      3Q     Max 
-7.8430 -3.2355  0.6887  2.7961  7.5627 

Coefficients:
                 Estimate Std. Error t value Pr(>|t|)    
(Intercept)      27.66645    7.65517   3.614 0.000785 ***
hrs_sueno         0.91668    0.44403   2.064 0.045040 *  
profesor_part     2.51800    1.29013   1.952 0.057502 .  
media_sem_pasado  0.20346    0.06421   3.169 0.002819 ** 
tiempo_est        1.85078    0.29820   6.207 1.83e-07 ***
asistencia        0.23836    0.05646   4.222 0.000123 ***
nivel_socioec     2.05662    0.43320   4.748 2.30e-05 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

Residual standard error: 4.411 on 43 degrees of freedom
Multiple R-squared:  0.6916,	Adjusted R-squared:  0.6485 
F-statistic: 16.07 on 6 and 43 DF,  p-value: 1.369e-09


## Pregunta 1.5 — Relación entre betas y ponderaciones del código *(6 pts.)*

Analice la relación entre los coeficientes estimados (β̂) y las ponderaciones reales usadas en el código del Anexo para generar las notas.



In [ ]:
# Pregunta 1.5: Comparación entre betas estimados y ponderaciones reales



# Sección 2: Wooldridge *(70 puntos)*

Para esta sección utilizaremos el paquete `wooldridge`, que contiene bases de datos clásicas de econometría.

In [11]:
# Instalar y cargar el paquete wooldridge (solo necesario la primera vez en Colab)
if (!require(wooldridge)) install.packages("wooldridge")
library(wooldridge)



Loading required package: wooldridge

Warning message in library(package, lib.loc = lib.loc, character.only = TRUE, logical.return = TRUE, :
“there is no package called ‘wooldridge’”
Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



---
## 2A. Base `wage1` *(30 puntos)*

El modelo de regresión poblacional a estimar es:

$$wage = \beta_0 + \beta_1\, educ + \beta_2\, exper + \beta_3\, tenure + u$$

Donde:
- `wage` = salario por hora (USD)
- `educ` = años de escolaridad
- `exper` = años de experiencia laboral
- `tenure` = años en el trabajo actual

In [12]:
# Cargar y limpiar la base wage1
data("wage1")
wage1 <- na.omit(wage1)


### Pregunta 2A.1 — Estimadores MCO e interpretación *(8 pts.)*

Estime el modelo completo e interprete los resultados.

In [18]:
# Pregunta 2A.1: Modelo de regresión múltiple con wage1
modelo_wage <- lm(wage ~ educ + exper + tenure, data =wage1)
summary(modelo_wage)
cat("\n--- Interpretación de Coeficientes ---\n")
cat("1. Intercepto: Representa el salario por hora esperado (wage) cuando los años de educación (educ), experiencia (exper) y permanencia en el trabajo actual (tenure) son cero. Esto es el punto de partida del modelo.",
    "\n")
cat("2. educ: Por cada año adicional de educación (manteniendo exper y tenure constantes), se espera que el salario por hora aumente en X unidades (el valor del coeficiente β_educ).",
    "\n")
cat("3. exper: Por cada año adicional de experiencia laboral (manteniendo educ y tenure constantes), se espera que el salario por hora aumente en X unidades (el valor del coeficiente β_exper).",
    "\n")
cat("4. tenure: Por cada año adicional en el trabajo actual (manteniendo educ y exper constantes), se espera que el salario por hora aumente en X unidades (el valor del coeficiente β_tenure)..\n")


Call:
lm(formula = wage ~ educ + exper + tenure, data = wage1)

Residuals:
    Min      1Q  Median      3Q     Max 
-7.6068 -1.7747 -0.6279  1.1969 14.6536 

Coefficients:
            Estimate Std. Error t value Pr(>|t|)    
(Intercept) -2.87273    0.72896  -3.941 9.22e-05 ***
educ         0.59897    0.05128  11.679  < 2e-16 ***
exper        0.02234    0.01206   1.853   0.0645 .  
tenure       0.16927    0.02164   7.820 2.93e-14 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

Residual standard error: 3.084 on 522 degrees of freedom
Multiple R-squared:  0.3064,	Adjusted R-squared:  0.3024 
F-statistic: 76.87 on 3 and 522 DF,  p-value: < 2.2e-16



--- Interpretación de Coeficientes ---
1. Intercepto: Representa el salario por hora esperado (wage) cuando los años de educación (educ), experiencia (exper) y permanencia en el trabajo actual (tenure) son cero. Esto es el punto de partida del modelo. 
2. educ: Por cada año adicional de educación (manteniendo exper y tenure constantes), se espera que el salario por hora aumente en X unidades (el valor del coeficiente β_educ). 
3. exper: Por cada año adicional de experiencia laboral (manteniendo educ y tenure constantes), se espera que el salario por hora aumente en X unidades (el valor del coeficiente β_exper). 
4. tenure: Por cada año adicional en el trabajo actual (manteniendo educ y exper constantes), se espera que el salario por hora aumente en X unidades (el valor del coeficiente β_tenure)..


### Pregunta 2A.2 — ¿Los signos son los esperados? *(10 pts.)*

Antes de ver los resultados, reflexione: ¿qué signo debería tener cada coeficiente económicamente?

In [ ]:
# Pregunta 2A.2: Revisar signos de los coeficientes
Se espera un signo positivo en la educación porque a mayor educación mayor salario, la experiencia también debe ser positiva, ya que, una mayor experiencia te da los recursus para un trabajo mejor, y por último se espera un signo positivo en tenure, esto debido a que el llevar un tiempo largo en el mismo trabajo genera reconocimiento y poder negociar salarios más altos.

### Pregunta 2A.3 — Comparar R² y R² ajustado *(12 pts.)*

Estime un segundo modelo usando solo `educ` y `tenure`, y compare el ajuste con el modelo completo.

In [22]:
# Pregunta 2A.3: Modelo reducido (sin exper)
lm(formula = wage ~ educ + tenure, data = wage1)
modelo_wage_reducido <- lm(wage ~ educ + tenure, data = wage1)
summary(modelo_wage1_reducido)
cat("\n--- Comparación de R-cuadrado ---\n")
r_squared_completo <- summary(modelo_wage)$r.squared
adj_r_squared_completo <- summary(modelo_wage)$adj.r.squared
r_squared_reducido <- summary(modelo_wage_reducido)$r.squared
adj_r_squared_reducido <- summary(modelo_wage_reducido)$adj.r.squared

cat("Modelo Completo (educ + exper + tenure):\n")
cat("  R-squared: ", round(r_squared_completo, 4), "\n")
cat("  Adjusted R-squared: ", round(adj_r_squared_completo, 4), "\n\n")

cat("Modelo Reducido (educ + tenure):\n")
cat("  R-squared: ", round(r_squared_reducido, 4), "\n")
cat("  Adjusted R-squared: ", round(adj_r_squared_reducido, 4), "\n")

cat("\nAnálisis:\n")
cat("El R-cuadrado mide la proporción de la varianza en la variable dependiente que es predecible a partir de las variables independientes. El R-cuadrado ajustado corrige el R-cuadrado en función del número de predictores en el modelo. Una disminución en el R-cuadrado ajustado al eliminar una variable sugiere que la variable eliminada aportaba poder explicativo relevante al modelo.")


Call:
lm(formula = wage ~ educ + tenure, data = wage1)

Coefficients:
(Intercept)         educ       tenure  
    -2.2216       0.5691       0.1896  



Call:
lm(formula = wage ~ educ + tenure, data = wage1)

Residuals:
    Min      1Q  Median      3Q     Max 
-8.1438 -1.7288 -0.6372  1.2575 14.7482 

Coefficients:
            Estimate Std. Error t value Pr(>|t|)    
(Intercept) -2.22162    0.64015   -3.47 0.000563 ***
educ         0.56914    0.04881   11.66  < 2e-16 ***
tenure       0.18958    0.01871   10.13  < 2e-16 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

Residual standard error: 3.092 on 523 degrees of freedom
Multiple R-squared:  0.3019,	Adjusted R-squared:  0.2992 
F-statistic: 113.1 on 2 and 523 DF,  p-value: < 2.2e-16



--- Comparación de R-cuadrado ---
Modelo Completo (educ + exper + tenure):
  R-squared:  0.3064 
  Adjusted R-squared:  0.3024 

Modelo Reducido (educ + tenure):
  R-squared:  0.3019 
  Adjusted R-squared:  0.2992 

Análisis:
El R-cuadrado mide la proporción de la varianza en la variable dependiente que es predecible a partir de las variables independientes. El R-cuadrado ajustado corrige el R-cuadrado en función del número de predictores en el modelo. Una disminución en el R-cuadrado ajustado al eliminar una variable sugiere que la variable eliminada aportaba poder explicativo relevante al modelo.

---
## 2B. Base `attend` *(40 puntos)*

En esta sección analizamos los determinantes del rendimiento en el examen final de un curso universitario.

### Pregunta 2B.1 — Cargar la base `attend` *(4 pts.)*

In [23]:
# Pregunta 2B.1: Cargar base attend
data("attend")
attend <- na.omit(attend)

### Pregunta 2B.2 — Selección de variables *(4 pts.)*

Subseleccione las siguientes variables:

| Variable | Descripción |
|---|---|
| `attend` | Clases asistidas de un total de 32 |
| `termGPA` | Promedio de notas durante el período |
| `priGPA` | Promedio acumulado antes del período |
| `ACT` | Puntaje en el examen ACT |
| `final` | **Variable dependiente** — puntaje del examen final |
| `hwrte` | Porcentaje de tareas entregadas |
| `frosh` | =1 si es estudiante de primer año |
| `soph` | =1 si es estudiante de segundo año |

In [ ]:
# Pregunta 2B.2: Subselección de variables
final, attend, priGPA

### Pregunta 2B.3 — Modelo de regresión múltiple completo *(10 pts.)*

Estime un modelo donde la variable dependiente es `final` y las independientes son todas las demás variables del subconjunto.

In [ ]:
# Pregunta 2B.3: Modelo completo con attend_new


### Pregunta 2B.4 — Bondad de ajuste *(6 pts.)*

Interprete el **R²** y el **R² ajustado** del modelo anterior.

In [ ]:
# Pregunta 2B.4: Extraer métricas de bondad de ajuste

### Pregunta 2B.5 — Modelo reducido (excluir variables no significativas) *(10 pts.)*

Genere un nuevo modelo excluyendo las variables con **p-value > 0.05** en el modelo anterior.

In [ ]:
# Identificar variables significativas (p-value <= 0.05)

In [ ]:
# Pregunta 2B.5: Modelo reducido (solo variables significativas)
# Variables significativas identificadas: termGPA, ACT, soph
# (ajuste según los resultados de su modelo)

### Pregunta 2B.6 — Comparación de modelos *(6 pts.)*

Compare el modelo completo y el modelo reducido en términos de **R² ajustado**.

In [ ]:
# Pregunta 2B.6: Comparación final de modelos